#### Data with some nulls

In [0]:
data = [
    (1, "A", None),
    (2, "B", 5000),
    (3, None, 7000),
    (4, "D", None)
]
df = spark.createDataFrame(data, ["id", "name", "salary"])
df1 = spark.createDataFrame(data, ["id", "name", "salary"])

In [0]:
df.display()
df.printSchema()

id,name,salary
1,A,null
2,B,5000
3,null,7000
4,D,null


root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)



#### isNull()/isNotNull()

In [0]:
df.filter(df.salary.isNull()).display()

id,name,salary
1,A,null
4,D,null


#### fillna()

In [0]:
df.fillna({"salary": 0}).display()

id,name,salary
1,A,0
2,B,5000
3,null,7000
4,D,0


#### dropna()

In [0]:
df.dropna().display()

id,name,salary
2,B,5000


#### coalesce()

In [0]:
from pyspark.sql.functions import coalesce, lit

df.withColumn("salary_new", coalesce(df.salary, lit(0))).display()

id,name,salary,salary_new
1,A,null,0
2,B,5000,5000
3,null,7000,7000
4,D,null,0


#### Student Task
##### Replace NULL salary with avg salary
##### Drop rows where name is NULL
##### Count NULL records

In [0]:
from pyspark.sql.functions import *
avg_salary = df.select(avg(col("salary"))).collect()[0][0]
df.fillna({"salary": avg_salary}).display()

id,name,salary,bonus,category,grade
1,A,1000,100.0,Low,B
2,B,5000,500.0,Low,B
3,null,7000,700.0,High,A
4,D,1000,100.0,Low,B


In [0]:
df.dropna(subset=["name"]).show()

+---+----+------+-----+--------+-----+
| id|name|salary|bonus|category|grade|
+---+----+------+-----+--------+-----+
|  1|   A|  1000|100.0|     Low|    B|
|  2|   B|  5000|500.0|     Low|    B|
|  4|   D|  1000|100.0|     Low|    B|
+---+----+------+-----+--------+-----+



In [0]:
df1.filter(
    col("id").isNull() | col("name").isNull() | col("salary").isNull()
).count()


3

##### Type Casting + Column Operations

##### Casting

In [0]:
df = df.withColumn("salary", df.salary.cast("int"))
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: integer (nullable = true)



##### Column Operations

In [0]:
from pyspark.sql.functions import col, when

df.select("id", "salary").display()

df.withColumn("bonus", col("salary") * 0.1).display()

df.withColumn("category",
    when(col("salary") > 6000, "High")
    .otherwise("Low")
).display()

id,salary
1,null
2,5000
3,7000
4,null


id,name,salary,bonus
1,A,null,null
2,B,5000,500.0
3,null,7000,700.0
4,D,null,null


id,name,salary,category
1,A,null,Low
2,B,5000,Low
3,null,7000,High
4,D,null,Low


In [0]:
# Step 1: Replace NULL salary with 1000
df = df.withColumn(
    "salary",
    when(col("salary").isNull(), 1000)
    .otherwise(col("salary"))
)

# Step 2: Add bonus column (10% of salary)
df = df.withColumn(
    "bonus",
    col("salary") * 0.1
)

# Step 3: Add category column
df = df.withColumn(
    "category",
    when(col("salary") > 6000, "High")
    .otherwise("Low")
)

# Final output
print("Final Data:")
df.display()

Final Data:


id,name,salary,bonus,category
1,A,1000,100.0,Low
2,B,5000,500.0,Low
3,null,7000,700.0,High
4,D,1000,100.0,Low


### Student Task
##### Create new column: tax = salary * 0.2
##### Create flag: is_high_salary
##### Rename column

In [0]:
df.withColumn("tax", col("salary") * 0.2).display()

id,name,salary,tax
1,A,null,null
2,B,5000,1000.0
3,null,7000,1400.0
4,D,null,null


In [0]:
df.withColumn(
    "is_high_salary",
    when(col("salary") > 6000, True).otherwise(False)
).display()

id,name,salary,is_high_salary
1,A,null,false
2,B,5000,false
3,null,7000,true
4,D,null,false


In [0]:
df.withColumnRenamed("salary", "salary_amount").display()

id,name,salary_amount
1,A,null
2,B,5000
3,null,7000
4,D,null


In [0]:
def grade(salary):
    if salary is None:
        return "Unknown"
    elif salary > 6000:
        return "A"
    else:
        return "B"

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

grade_udf = udf(grade, StringType())

In [0]:
df = df.withColumn("grade", grade_udf(df.salary))
df.display()

id,name,salary,bonus,category,grade
1,A,1000,100.0,Low,B
2,B,5000,500.0,Low,B
3,null,7000,700.0,High,A
4,D,1000,100.0,Low,B


#### Same logic without UDF

In [0]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "grade",
    when(col("salary").isNull(), "Unknown")
    .when(col("salary") > 6000, "A")
    .otherwise("B")
)